In [1]:
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
documents = {
    "doc1": "This is the first document.",
    "doc2": "This document is the second document.",
    "doc3": "And this is the third one.",
    "doc4": "Is this the first document?",
}

In [3]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [token for token in tokens if token not in stopwords.words("english") and token not in string.punctuation]
    return " ".join(tokens)

In [4]:
preprocessed_docs = {doc_id: preprocess_text(doc) for doc_id, doc in documents.items()}

In [5]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_docs.values())

In [6]:
# Calculate cosine similarity between query and documents
def search(query, tfidf_matrix, tfidf_vectorizer):

    # Preprocess query
    processed_query = preprocess_text(query)

    # Convert query into TF-IDF vector
    query_vector = tfidf_vectorizer.transform([processed_query])

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Store results
    results = [
        (doc_id, documents[doc_id], score)
        for doc_id, score in zip(documents.keys(), similarity_scores)
    ]

    # Sort results by similarity score
    return sorted(results, key=lambda x: x[2], reverse=True)

In [10]:
# Get input from user
query = input("Enter your query: ")

# Perform search
search_results = search(
    query,
    tfidf_matrix,
    tfidf_vectorizer
)

Enter your query: first document


In [11]:
# Display search results
print("Query:", query)

Query: first document


In [12]:
for i, result in enumerate(search_results, start=1):

    print(f"\nRank: {i}")
    print("Document ID:", result[0])
    print("Document:", result[1])
    print("Similarity Score:", result[2])
    print("----------------------")

# Get the highest rank cosine score
highest_rank_score = max(
    result[2] for result in search_results
)
print("The highest rank cosine score is:",
      highest_rank_score)


Rank: 1
Document ID: doc1
Document: This is the first document.
Similarity Score: 1.0
----------------------

Rank: 2
Document ID: doc4
Document: Is this the first document?
Similarity Score: 1.0
----------------------

Rank: 3
Document ID: doc2
Document: This document is the second document.
Similarity Score: 0.4953423567447137
----------------------

Rank: 4
Document ID: doc3
Document: And this is the third one.
Similarity Score: 0.0
----------------------
The highest rank cosine score is: 1.0
